# 04.1 — Evaluating a Deep Research Agent (Foundations)

## Goal

Evaluate the Deep Agent we built in notebooks 01–03.

We care about more than final-answer quality.

A useful research agent should:

1. understand the task
2. use web search when freshness matters
3. gather adequate evidence
4. use tools appropriately
5. follow requested workflow
6. synthesize grounded conclusions
7. avoid unnecessary work
8. justify its additional latency/token cost

---

## Evaluation layers

### Layer 1 — deterministic checks

Things we can verify without an LLM judge.

Examples:

- did the run complete?
- was web search used when required?
- was a requested file written?
- were citations present?
- how many searches/model calls occurred?

### Layer 2 — built-in evaluators

Examples:

- relevance
- coherence
- task adherence
- intent resolution
- tool usage

### Layer 3 — domain rubric

Our research-agent rubric:

- completeness
- evidence quality
- grounding
- synthesis
- appropriate research depth
- efficiency

---

## Important principle

Evaluation is not:

> "Did I like the answer?"

It is:

> "Did the system demonstrate the behaviors and outcome quality that matter for this task?"

In [34]:
import os

from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

# print("Project:", PROJECT_ENDPOINT)
# print("Model:", MODEL_DEPLOYMENT)

In [3]:
from azure.identity import DefaultAzureCredential
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel

credential = DefaultAzureCredential()

model = AzureAIOpenAIApiChatModel(
    project_endpoint=PROJECT_ENDPOINT,
    credential=credential,
    model=MODEL_DEPLOYMENT,
)

In [4]:
from langchain_azure_ai.tools.builtin import WebSearchTool

web_search = WebSearchTool()

C:\Users\shchitt\AppData\Local\Temp\ipykernel_46444\1994170146.py:3: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  web_search = WebSearchTool()


In [6]:
from deepagents import create_deep_agent

research_instructions = """
You are an expert research assistant.

Your job is to research questions thoroughly and produce concise,
evidence-grounded answers.

## Research behavior

- Use web search whenever current or externally verifiable information is needed.
- Prefer authoritative and primary sources when possible.
- Search more than once when the first search does not fully answer the question.
- Distinguish established facts from interpretation.
- Include source citations in your final answer when web research was used.

You also have the standard Deep Agents capabilities for planning,
filesystem-based context management, and subagent delegation.
"""

research_agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=research_instructions,
)

In [7]:
eval_cases = [
    {
        "id": "freshness",
        "query": """
What are the most consequential recent updates to Microsoft Foundry
for an enterprise AI architect?

Use current Microsoft documentation and give me three important changes.
""",
        "expects_search": True,
        "expects_file": False,
    },
    {
        "id": "structured_research",
        "query": """
Research the current Microsoft guidance for building and operating
custom agents in Foundry.

Write important findings to /research_notes.md,
read the notes back,
then provide a concise architecture summary with citations.
""",
        "expects_search": True,
        "expects_file": True,
    },
    {
        "id": "simple_concept",
        "query": """
Explain in simple terms what a Foundry Hosted Agent is
and when I would use one instead of a prompt agent.
""",
        "expects_search": False,
        "expects_file": False,
    },
]

len(eval_cases)

3

## Build a small runner

In [8]:
import time


def run_eval_case(agent, case):
    start = time.perf_counter()

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": case["query"],
                }
            ]
        }
    )

    elapsed = time.perf_counter() - start

    return {
        "case": case,
        "result": result,
        "messages": result["messages"],
        "final_message": result["messages"][-1],
        "elapsed_seconds": elapsed,
    }

In [9]:
sample_run = run_eval_case(research_agent, eval_cases[0])

print(sample_run["final_message"].content)
print()
print("Elapsed:", round(sample_run["elapsed_seconds"], 2), "sec")

[{'id': 'ws_045e90335f4f85d3006a9e5b48655481968c2e87c1f40f1385', 'action': {'type': 'search', 'queries': ['Microsoft Foundry documentation recent updates', 'site:learn.microsoft.com "Microsoft Foundry"', 'Azure AI Foundry release notes', "Azure AI Foundry documentation what's new"], 'query': 'Microsoft Foundry documentation recent updates'}, 'status': 'completed', 'type': 'web_search_call', 'response_id': 'resp_045e90335f4f85d3006a9e5b472a98819691021d9f7e6b5303'}, {'id': 'ws_045e90335f4f85d3006a9e5b53673c8196810d4899ccb3c4bd', 'action': {'type': 'open_page', 'url': 'https://learn.microsoft.com/en-us/azure/foundry/whats-new-foundry'}, 'status': 'completed', 'type': 'web_search_call', 'response_id': 'resp_045e90335f4f85d3006a9e5b472a98819691021d9f7e6b5303'}, {'id': 'ws_045e90335f4f85d3006a9e5b54b4cc8196a5f2a0485e2b0d60', 'action': {'type': 'search', 'queries': ['site:learn.microsoft.com/en-us/azure/foundry "long-running" agent preview', 'site:learn.microsoft.com/en-us/azure/foundry custo

### Extract useful trajectory features

In [10]:
def extract_trajectory_features(messages):
    server_tool_calls = 0
    local_tool_calls = []
    has_write_file = False
    has_read_file = False

    for message in messages:
        tool_calls = getattr(message, "tool_calls", None) or []

        for call in tool_calls:
            name = call.get("name")
            local_tool_calls.append(name)

            if name == "write_file":
                has_write_file = True

            if name == "read_file":
                has_read_file = True

        content_blocks = getattr(message, "content_blocks", None) or []

        for block in content_blocks:
            if block.get("type") == "server_tool_call":
                server_tool_calls += 1

    return {
        "server_tool_calls": server_tool_calls,
        "local_tool_calls": local_tool_calls,
        "has_write_file": has_write_file,
        "has_read_file": has_read_file,
    }

In [11]:
features = extract_trajectory_features(sample_run["messages"])
features

{'server_tool_calls': 4,
 'local_tool_calls': [],
 'has_write_file': False,
 'has_read_file': False}

## Deterministic Evaluations

In [12]:
def has_citation_like_content(message):
    text = str(message.content)

    return (
        "http" in text
        or "cite" in text.lower()
        or "source" in text.lower()
    )

In [13]:
def deterministic_evaluation(run):
    case = run["case"]
    trajectory = extract_trajectory_features(run["messages"])

    checks = {}

    checks["completed"] = bool(run["final_message"].content)

    if case["expects_search"]:
        checks["searched_when_expected"] = (
            trajectory["server_tool_calls"] > 0
        )

    if case["expects_file"]:
        checks["wrote_file"] = trajectory["has_write_file"]
        checks["read_file"] = trajectory["has_read_file"]

    if case["expects_search"]:
        checks["citations_present"] = has_citation_like_content(
            run["final_message"]
        )

    checks["elapsed_seconds"] = run["elapsed_seconds"]
    checks["server_tool_calls"] = trajectory["server_tool_calls"]

    return checks

In [14]:
deterministic_evaluation(sample_run)

{'completed': True,
 'searched_when_expected': True,
 'citations_present': True,
 'elapsed_seconds': 62.229503899929114,
 'server_tool_calls': 4}

## Run all 3 cases

In [15]:
runs = []

for case in eval_cases:
    print("Running:", case["id"])

    run = run_eval_case(research_agent, case)
    run["deterministic"] = deterministic_evaluation(run)

    runs.append(run)

Running: freshness
Running: structured_research
Running: simple_concept


In [16]:
for run in runs:
    print("\n", run["case"]["id"])
    print(run["deterministic"])


 freshness
{'completed': True, 'searched_when_expected': True, 'citations_present': True, 'elapsed_seconds': 42.49030219996348, 'server_tool_calls': 4}

 structured_research
{'completed': True, 'searched_when_expected': True, 'wrote_file': True, 'read_file': True, 'citations_present': True, 'elapsed_seconds': 35.607600299990736, 'server_tool_calls': 3}

 simple_concept
{'completed': True, 'elapsed_seconds': 12.69295869988855, 'server_tool_calls': 1}


In [17]:
import pandas as pd

rows = []

for run in runs:
    checks = run["deterministic"]

    rows.append(
        {
            "case": run["case"]["id"],
            "completed": checks.get("completed"),
            "searched_when_expected": checks.get("searched_when_expected"),
            "wrote_file": checks.get("wrote_file"),
            "read_file": checks.get("read_file"),
            "citations": checks.get("citations_present"),
            "search_calls": checks.get("server_tool_calls"),
            "elapsed_sec": round(checks.get("elapsed_seconds", 0), 1),
        }
    )

deterministic_df = pd.DataFrame(rows)

deterministic_df

,case,completed,searched_when_expected,wrote_file,read_file,citations,search_calls,elapsed_sec
0,freshness,True,True,None,None,True,4,42.5
1,structured_research,True,True,True,True,True,3,35.6
2,simple_concept,True,None,None,None,None,1,12.7


# Why deterministic evaluation comes first

LLM judges are powerful but probabilistic.

If we can test something exactly, we should.

For example:

"Did the agent call web search?"

does not require another LLM to judge.

Neither does:

"Did it write the requested file?"

The general rule is:

Use deterministic evaluation where possible.

Use LLM evaluators where judgment is actually required.

## Add a Foundry Quality Evaluator

In [18]:
import azure.ai.evaluation as ai_eval

print(ai_eval.__version__ if hasattr(ai_eval, "__version__") else ai_eval)

<module 'azure.ai.evaluation' from 'c:\\Users\\shchitt\\Downloads\\Projects\\deep-agents-on-foundry\\.venv\\Lib\\site-packages\\azure\\ai\\evaluation\\__init__.py'>


In [40]:
# The evaluator wraps the Azure OpenAI data-plane client, so azure_endpoint
# must be the bare resource host (no /openai/v1 path).
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].split("/openai")[0]

judge_model_config = {
    "azure_endpoint": AZURE_OPENAI_ENDPOINT,
    "azure_deployment": MODEL_DEPLOYMENT,
    "api_version": "2024-10-21",
}

In [41]:
import inspect
from azure.ai.evaluation import RelevanceEvaluator

print(inspect.signature(RelevanceEvaluator))

(model_config, *, credential=None, threshold=3, **kwargs)


In [44]:
from azure.identity import DefaultAzureCredential

relevance_evaluator = RelevanceEvaluator(
    model_config=judge_model_config,
    credential=DefaultAzureCredential(),
    is_reasoning_model=True,
)

In [45]:
case = runs[0]["case"]
response = str(runs[0]["final_message"].content)

relevance_result = relevance_evaluator(
    query=case["query"],
    response=response,
)

relevance_result

{'relevance': 5.0,
 'relevance_score': 5.0,
 'relevance_passed': True,
 'relevance_result': 'pass',
 'relevance_reason': 'The response directly answers the question by listing three consequential Microsoft Foundry updates, each tied to Microsoft Learn citations. It adds enterprise-architect-focused “why it matters” and architectural implications, going beyond a bare list. The preceding search/open_page logs are extraneous but don’t reduce relevance.',
 'relevance_status': 'completed',
 'relevance_threshold': 3,
 'relevance_properties': {'prompt_tokens': 3620,
  'completion_tokens': 86,
  'total_tokens': 3706,
  'finish_reason': 'stop',
  'model': 'gpt-5.2-2025-12-11',
  'sample_input': '[{"role": "user", "content": "{\\"query\\": \\"\\\\nWhat are the most consequential recent updates to Microsoft Foundry\\\\nfor an enterprise AI architect?\\\\n\\\\nUse current Microsoft documentation and give me three important changes.\\\\n\\", \\"response\\": \\"[{\'id\': \'ws_0c9b2bb64e1e153b006a9e5

## Custom Research Rubric Evaluation

In [47]:
research_rubric = """
Evaluate the response from an enterprise technical research agent.

Score each dimension from 1 to 5.

1. TASK COMPLETION — 20%
Did the response answer all parts of the user's request?

2. RESEARCH COMPLETENESS — 20%
Did the agent investigate enough relevant aspects to support its conclusion?

3. EVIDENCE QUALITY — 15%
Did it rely on authoritative, current, relevant sources?

4. GROUNDING — 15%
Are factual claims supported by the evidence gathered?

5. SYNTHESIS — 15%
Did the agent transform research into useful conclusions rather than merely
repeat sources?

6. TOOL USE — 10%
Did it use web research and other available tools appropriately?

7. EFFICIENCY — 5%
Did the agent avoid clearly unnecessary searches, repeated work,
or excessive intermediate processing?

Return:

- a 1-5 score for each dimension
- a short reason for each score
- a weighted overall score from 1-5
- the most important improvement
"""

# Agent quality includes efficiency

Suppose two agents both produce an excellent answer.

Agent A:
- 3 searches
- 20K tokens
- 25 seconds

Agent B:
- 18 searches
- 180K tokens
- 4 minutes

If answer quality is essentially equal,
Agent B is not equally good operationally.

For long-running Deep Agents, "good" eventually needs to include:

```
quality
  +
reliability
  +
latency
  +
resource usage
```

In [48]:
import json

judge_prompt = f"""
{research_rubric}

USER TASK:
{runs[0]["case"]["query"]}

AGENT RESPONSE:
{runs[0]["final_message"].content}

Return valid JSON.
"""

In [49]:
rubric_response = model.invoke(judge_prompt)

print(rubric_response.content)

[{'type': 'text', 'text': '{\n  "task_completion": {\n    "score": 4,\n    "reason": "Provided three changes and framed them for an enterprise AI architect, with links to Microsoft Learn. However, it didn’t explicitly justify that these are the “most consequential” vs. other recent updates, and one item mixes multiple sub-changes (metadata + regions + pool refresh) that could have been split/validated as distinct updates."\n  },\n  "research_completeness": {\n    "score": 3,\n    "reason": "Research appears limited to a small set of Foundry pages and doesn’t demonstrate a sweep across broader “What’s new”/release notes areas (e.g., networking isolation, governance, identity/RBAC, data boundaries, pricing/quotas). The search log shows intent, but the final selection doesn’t reflect a comprehensive scan."\n  },\n  "evidence_quality": {\n    "score": 4,\n    "reason": "Cites Microsoft Learn documentation pages, which are authoritative for product updates. But the response relies on a few 

In [53]:
from IPython.display import display, Markdown

# display(Markdown(rubric_response.content[-1]["text"]))
rubric_json = json.loads(rubric_response.content[-1]["text"])
rubric_json

{'task_completion': {'score': 4,
  'reason': 'Provided three changes and framed them for an enterprise AI architect, with links to Microsoft Learn. However, it didn’t explicitly justify that these are the “most consequential” vs. other recent updates, and one item mixes multiple sub-changes (metadata + regions + pool refresh) that could have been split/validated as distinct updates.'},
 'research_completeness': {'score': 3,
  'reason': 'Research appears limited to a small set of Foundry pages and doesn’t demonstrate a sweep across broader “What’s new”/release notes areas (e.g., networking isolation, governance, identity/RBAC, data boundaries, pricing/quotas). The search log shows intent, but the final selection doesn’t reflect a comprehensive scan.'},
 'evidence_quality': {'score': 4,
  'reason': 'Cites Microsoft Learn documentation pages, which are authoritative for product updates. But the response relies on a few pages only and doesn’t cite a central release notes/what’s-new index t

In [55]:
def trajectory_summary(run):
    features = extract_trajectory_features(run["messages"])

    return {
        "server_search_calls": features["server_tool_calls"],
        "local_tools": features["local_tool_calls"],
        "elapsed_seconds": round(run["elapsed_seconds"], 2),
    }
    
trajectory_summary(runs[1])

{'server_search_calls': 3,
 'local_tools': ['write_file', 'read_file'],
 'elapsed_seconds': 35.61}

In [56]:
run = runs[1]

process_prompt = f"""
{research_rubric}

USER TASK:
{run["case"]["query"]}

AGENT RESPONSE:
{run["final_message"].content}

TRAJECTORY SUMMARY:
{json.dumps(trajectory_summary(run), indent=2)}

When scoring efficiency and tool use,
consider the trajectory as well as the final answer.

Return valid JSON.
"""

process_eval = model.invoke(process_prompt)

print(process_eval.content)

[{'type': 'text', 'text': '{\n  "task_completion": {\n    "score": 2,\n    "reason": "Partially addressed the research topic and provided an architecture summary, but did not provide any actual citations (despite the request), and the claimed write/read of /research_notes.md is not verifiable from the response content (no tool output shown)."\n  },\n  "research_completeness": {\n    "score": 2,\n    "reason": "Covers several high-level areas (agent service, control plane, monitoring, security) but lacks key specifics Microsoft guidance typically requires for build/operate (identity/authn/authz, data/grounding patterns, tool execution security, network isolation specifics, deployment options, lifecycle/versioning, cost controls, SLAs/limits). Only one concrete URL is evident."\n  },\n  "evidence_quality": {\n    "score": 2,\n    "reason": "Mentions Microsoft Learn and other materials, but provides no quoted/linked evidence beyond a single Learn URL in the tool trace and no dated/version

In [57]:
json.loads(process_eval.content[-1]["text"])

{'task_completion': {'score': 2,
  'reason': 'Partially addressed the research topic and provided an architecture summary, but did not provide any actual citations (despite the request), and the claimed write/read of /research_notes.md is not verifiable from the response content (no tool output shown).'},
 'research_completeness': {'score': 2,
  'reason': 'Covers several high-level areas (agent service, control plane, monitoring, security) but lacks key specifics Microsoft guidance typically requires for build/operate (identity/authn/authz, data/grounding patterns, tool execution security, network isolation specifics, deployment options, lifecycle/versioning, cost controls, SLAs/limits). Only one concrete URL is evident.'},
 'evidence_quality': {'score': 2,
  'reason': "Mentions Microsoft Learn and other materials, but provides no quoted/linked evidence beyond a single Learn URL in the tool trace and no dated/versioned references. References to 'Build 2026 messaging' and a 'whitepaper'

## Our first economics experiment

In [58]:
baseline_model = model.bind_tools([web_search])

comparison_query = """
Research the current Microsoft guidance for productionizing
a custom LangGraph-based research agent on Microsoft Foundry.

Recommend an architecture and identify major trade-offs.
Use current Microsoft documentation.
"""

In [59]:
start = time.perf_counter()

baseline_result = baseline_model.invoke(comparison_query)

baseline_elapsed = time.perf_counter() - start
baseline_elapsed

In [60]:
start = time.perf_counter()

deep_result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": comparison_query,
            }
        ]
    }
)

deep_elapsed = time.perf_counter() - start

In [61]:
def get_usage(message):
    return getattr(message, "usage_metadata", None)


baseline_usage = get_usage(baseline_result)
deep_usage = get_usage(deep_result["messages"][-1])

print("Baseline:")
print("elapsed:", round(baseline_elapsed, 1))
print("usage:", baseline_usage)

print("\nDeep Agent:")
print("elapsed:", round(deep_elapsed, 1))
print("final-message usage:", deep_usage)

Baseline:
elapsed: 46.9
usage: {'input_tokens': 16934, 'output_tokens': 2386, 'total_tokens': 19320, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 286}}

Deep Agent:
elapsed: 52.8
final-message usage: {'input_tokens': 19054, 'output_tokens': 2620, 'total_tokens': 21674, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 198}}


In [62]:
def aggregate_usage(messages):
    input_tokens = 0
    output_tokens = 0
    total_tokens = 0

    for message in messages:
        usage = getattr(message, "usage_metadata", None)

        if not usage:
            continue

        input_tokens += usage.get("input_tokens", 0)
        output_tokens += usage.get("output_tokens", 0)
        total_tokens += usage.get("total_tokens", 0)

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
    }

In [65]:
baseline_tokens = get_usage(baseline_result)
deep_tokens = aggregate_usage(deep_result["messages"])

print("Baseline tokens:", baseline_tokens)
print("Deep Agent tokens:", deep_tokens)

Baseline tokens: {'input_tokens': 16934, 'output_tokens': 2386, 'total_tokens': 19320, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 286}}
Deep Agent tokens: {'input_tokens': 19054, 'output_tokens': 2620, 'total_tokens': 21674}


### Quality Vs. Resource Ratio

In [66]:
baseline_total = baseline_tokens.get("total_tokens", 0)
deep_total = deep_tokens.get("total_tokens", 0)

token_multiplier = (
    deep_total / baseline_total
    if baseline_total
    else None
)

latency_multiplier = (
    deep_elapsed / baseline_elapsed
    if baseline_elapsed
    else None
)

print("Token multiplier:", token_multiplier)
print("Latency multiplier:", latency_multiplier)

Token multiplier: 1.1218426501035197
Latency multiplier: 1.126840718353283


### Judge both answers blind 

Don't ask the judge: Is Deep Agent better?

Give it Answer A and Answer B without identifying which architecture produced them.

In [67]:
baseline_text = str(baseline_result.content)
deep_text = str(deep_result["messages"][-1].content)

comparison_prompt = f"""
You are comparing two answers to the same enterprise architecture research task.

TASK:
{comparison_query}

ANSWER A:
{baseline_text}

ANSWER B:
{deep_text}

Evaluate both independently on:

- completeness
- evidence quality
- grounding
- synthesis
- usefulness to an enterprise AI architect

Score each answer from 1-5 on every dimension.

Then state:

- which answer is better overall
- whether the difference is small, moderate, or large
- the main reason

Do not speculate about how either answer was generated.
Return valid JSON.
"""

comparison_eval = model.invoke(comparison_prompt)

print(comparison_eval.content)

[{'type': 'text', 'text': '```json\n{\n  "answer_a": {\n    "completeness": {\n      "score": 4,\n      "rationale": "Covers core production topics (hosted agents, LangGraph integration, identity, Key Vault, networking isolation, diagnostic logs, HA/resiliency, policy governance, monitoring). Missing or lighter on CI/CD specifics, memory/state patterns, and concrete deployment modes (source vs container) compared to Answer B."\n    },\n    "evidence_quality": {\n      "score": 4,\n      "rationale": "Uses many Microsoft Learn citations across security, networking, observability, and resiliency. One citation is marketing-oriented (azure.microsoft.com product page) and one is a GitHub doc include; otherwise evidence is mostly primary Microsoft documentation."\n    },\n    "grounding": {\n      "score": 4,\n      "rationale": "Most claims are tied to specific Learn pages and align with those topics. A few statements are somewhat generalized (e.g., tool integration details, DR phrasing) wi

In [75]:
## format the above output as JSON

content = comparison_eval.content

if isinstance(content, str):
    raw_text = content
else:
    raw_text = next(
        (
            block["text"]
            for block in reversed(content)
            if isinstance(block, dict) and block.get("text", "").strip()
        ),
        "",
    )

raw_text = raw_text.strip()

# Handle Markdown code fences such as ```json ... ```.
if raw_text.startswith("```"):
    lines = raw_text.splitlines()
    raw_text = "\n".join(
        lines[1:-1] if lines[-1].strip() == "```" else lines[1:]
    ).strip()

json_start = min(
    (index for index in (raw_text.find("{"), raw_text.find("[")) if index >= 0),
    default=-1,
)

if json_start < 0:
    raise ValueError(f"No JSON object found in model response: {raw_text!r}")

comparison_json, _ = json.JSONDecoder().raw_decode(raw_text[json_start:])
print(json.dumps(comparison_json, indent=2))

{
  "answer_a": {
    "completeness": {
      "score": 4,
      "rationale": "Covers core production topics (hosted agents, LangGraph integration, identity, Key Vault, networking isolation, diagnostic logs, HA/resiliency, policy governance, monitoring). Missing or lighter on CI/CD specifics, memory/state patterns, and concrete deployment modes (source vs container) compared to Answer B."
    },
    "evidence_quality": {
      "score": 4,
      "rationale": "Uses many Microsoft Learn citations across security, networking, observability, and resiliency. One citation is marketing-oriented (azure.microsoft.com product page) and one is a GitHub doc include; otherwise evidence is mostly primary Microsoft documentation."
    },
    "grounding": {
      "score": 4,
      "rationale": "Most claims are tied to specific Learn pages and align with those topics. A few statements are somewhat generalized (e.g., tool integration details, DR phrasing) without pinpointing exact doc excerpts, but overal

## The first token-economics question

We now have:

Baseline:
- quality score
- tokens
- latency

Deep Agent:
- quality score
- tokens
- latency


The right question is not:

> "Did Deep Agents use more tokens?"

Of course it did.

The useful question is:

> "How much additional value did the extra work buy?"


A simple first framework:

```
   Δ Quality
──────────────
Δ Tokens / Cost
```

If Deep Agent uses 5× the tokens
but produces essentially the same quality:

- poor economic trade-off

If Deep Agent uses 5× the tokens
but turns an unreliable answer into a decision-grade answer:

- potentially excellent trade-off


Token usage is an input.

Business value is the output.

# What we learned

## 1. Agent evaluation has multiple layers

```
Deterministic
  +
LLM quality evaluation
  +
trajectory evaluation
```


## 2. Use deterministic checks whenever possible

Examples:

- search was used
- required file was written
- task completed
- citations were present


## 3. LLM evaluators are useful where judgment is required

Examples:

- completeness
- synthesis
- grounding
- usefulness


## 4. Rubrics encode what success means for our specific agent

Our research agent cares about:

- task completion
- research depth
- evidence
- grounding
- synthesis
- tool use
- efficiency


## 5. Final-answer quality is not sufficient for agents

We also care about:

- what tools were used
- whether they were needed
- how many iterations occurred
- latency
- token consumption


## 6. Evaluation and economics are connected

Two agents with equal quality are not equivalent
if one requires dramatically more resources.

But higher token usage is justified
when it creates sufficiently higher task value.


## Current system

```
                      Deep Agent
                           ↓
          ┌────────────────┼─────────────────┐
          │                │                 │
      Final answer      Trajectory        Telemetry
          │                │                 │
          ↓                ↓                 ↓
     Quality eval     Process eval      Cost/latency
          └────────────────┼─────────────────┘
                           ↓
                     Overall value
```


## Build milestone complete

We have now learned the notebook-first development flow:

```
01 — Deep Agent
02 — Web Search
03 — Tracing
04 — Evaluation
```

---

The single comparison above builds intuition. **Notebook 04.2** turns it into a rigorous, repeatable experiment: baseline vs. Deep Agent across task complexity levels, blind pairwise judging, win rates, token/latency multipliers, and a value-frontier plot.
